# Loading Data

In [ ]:
import os

base_path = "/kaggle/input/deepandhandcrafted"

for root, dirs, files in os.walk(base_path):
    for name in files:
        print(os.path.join(root, name))


In [ ]:
import pandas as pd

df_deep = pd.read_csv("/kaggle/input/deep-features/deep_features.csv")
df_hand = pd.read_csv("/kaggle/input/handcrafted-features/Handcrafted_Features_flair.csv")

print(df_deep.shape, df_hand.shape)
print(df_deep.columns[:5])
print(df_hand.columns[:5])


In [ ]:

feature_cols = [col for col in df_deep.columns if col not in ['id', 'label']]
df_deep_agg = df_deep.groupby('id')[feature_cols].mean().reset_index()

print("Aggregated deep shape:", df_deep_agg.shape)

df_fused = pd.merge(df_deep_agg, df_hand, on='id', how='inner')

print("Fused feature matrix shape:", df_fused.shape) 

X_fusion = df_fused.drop(columns=['id','label'])
y_fusion = df_fused['label']

In [ ]:
len(X_fusion.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_fusion)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif"], # Matches Times New Roman
    "font.size": 10,                    # Standard for most papers
    "axes.labelsize": 10,
    "xtick.labelsize": 8,              
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "text.usetex": False,               
    "pdf.fonttype": 42,               
    "ps.fonttype": 42
})

X = X_scaled
y = df_hand['label']


# SCREE PLOT

In [ ]:

# 1. PCA and Scree Plot
pca = PCA()
pca.fit(X)
explained_var = pca.explained_variance_ratio_

plt.figure(figsize=(8,5))
plt.plot(np.arange(1, len(explained_var) + 1), explained_var, marker='o')
plt.xlabel("Number of PCA Components")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot: Explained Variance per Component")
plt.grid()
plt.show()


# Cumulative Explained Variance Table

In [ ]:
cum_var = np.cumsum(explained_var)
table = []
for n in [1, 5, 10, 15, 20, 50, 100,150,200 ,300,400,500,600,700,800,900,1000,len(explained_var)]:
    if n <= len(cum_var):
        table.append([n, cum_var[n-1]])
cum_table = pd.DataFrame(table, columns=['Components', 'Cumulative Variance'])
print(cum_table)


# PCA Performance plot

In [ ]:
# for selecting n_components
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt


n_splits = 5

train_fold_limit = int(X_fusion.shape[0] * ((n_splits - 1) / n_splits))
max_possible_comp = min(train_fold_limit, X_fusion.shape[1])

print(f"Scanning up to {max_possible_comp} components (Safety Limit)...")


components_tested = [] 
scores = []

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for n_comp in range(1, max_possible_comp, 5):
    pipe = Pipeline([
        ('pca', PCA(n_components=n_comp)),
        ('clf', LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, n_jobs=-1))
    ])
    
    try:
        cv_scores = cross_val_score(pipe, X_fusion, y_fusion, cv=skf, scoring='accuracy', n_jobs=-1)
        mean_acc = np.mean(cv_scores)
        
        scores.append(mean_acc)
        components_tested.append(n_comp)
        
    except Exception as e:
        print(f"Error at n={n_comp}: {e}")

plt.figure(figsize=(8,5))
plt.plot(components_tested, scores, marker='o')
plt.xlabel('Number of PCA Components')
plt.ylabel('Classification Accuracy')
plt.title('Logistic Regression Accuracy vs Number of PCA Components')
plt.grid()
plt.show()